In [0]:
#Create Schema
from pyspark.sql.types import StringType, StructType, StructField
schema = StructType ([
    StructField("Interaction_Id", StringType(), True),
    StructField("Agent_Id", StringType(), True),
    StructField("Interaction_Type", StringType(), True),
    StructField("Survey_Score", StringType(), True),
    StructField("Interaction_Date", StringType(), True),
    StructField("Verified_PII", StringType(), True)
])

#Insert Data
bpo_data = [
    ("INT-001", "Melany Rua", "Tech Support", "95%", "2026/05/12", "  yes "),
    ("INT-002", "Tamara Solorzano", "Billing", " 80 ", "13-05-2026", "no"),
    ("INT-003", "Melany Rua", "Tech Support", "csat_fail", "2026-05-14 00:00:00", "Y"),
    ("INT-004", "Melany Rua", "Tech Support", "95%", "2026/05/12", "  yes "), # Duplicado exacto
    ("INT-005", "Kevin Velez", "Tech Support", None, "2026/05/15", "PENDING"),
    ("INT-006", "Tamara Solorzano", "Billing", "100%", "Invalida_Date", "n"),
    ("INT-007", "Anónimo", "Tech Support", " 75 ", "2026/05/15", "   ") # Espacios vacíos en PII
]

#Create DataFrame
bpo_data_raw = spark.createDataFrame(bpo_data, schema)

#Create view for SQL consultation
bpo_data_raw.createOrReplaceTempView("bpo_data_raw")

#Display data
display(bpo_data_raw)

In [0]:
%sql
    
CREATE OR REPLACE TABLE bpo_data_cleaned AS (
SELECT
    Interaction_Id,
    Agent_Id,
    Interaction_Type,
    TRY_CAST(trim(regexp_replace(Survey_Score, '[% ]', '')) AS INT) AS Survey_Score,
    COALESCE(
        TRY_TO_DATE(Interaction_Date, 'yyyy/MM/dd'),
        TRY_TO_DATE(Interaction_Date, 'dd-MM-yyyy'),
        TRY_TO_DATE(Interaction_Date, 'yyyy-MM-dd')
    ) AS Interaction_Date,
    CASE
        WHEN TRIM(LOWER(Verified_PII)) IN ('yes', 'y') THEN TRUE
        WHEN TRIM(LOWER(Verified_PII)) IN ('no', 'n') THEN FALSE
        ELSE NULL
    END AS Verified_PII
FROM
    bpo_data_raw
);

In [0]:
%sql
SELECT * FROM bpo_data_cleaned


In [0]:
%sql
--Drop nulls from survey_score
CREATE OR REPLACE TABLE bpo_data_cleaned_nulls AS (
SELECT 
    Interaction_Id,
    Agent_Id,
    COALESCE(Interaction_Type, 'na') AS Interaction_Type,
    COALESCE(Survey_Score, 0) AS Survey_Score,
    Interaction_Date,
    Verified_PII
FROM
    bpo_data_cleaned
)

In [0]:
%sql
SELECT * FROM bpo_data_cleaned_nulls

In [0]:
%sql
--Display schema of cleaned data
DESCRIBE bpo_data_cleaned_nulls